# gulf-buoy-etl — Pipeline Walkthrough

This notebook walks through one full ETL cycle using the bundled sample fixtures (no network calls). It shows how raw NDBC/TABS text becomes a CF-1.8 NetCDF archive with QC flags and SHA-256 fixity.

**Sections:**
1. Parse raw NDBC text
2. Validate (range / timestamp / gap)
3. Normalize units
4. Write daily NetCDF + SHA-256 sidecar
5. Inspect the produced NetCDF
6. Render the QC report

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

from gbe.config import DEFAULT_STATIONS
from gbe.sources import parse_ndbc_text
from gbe.transform import normalize_units, write_daily_netcdf
from gbe.validation import validate_dataframe

## 1. Parse raw NDBC text

In [ ]:
station = next(s for s in DEFAULT_STATIONS if s.id == '42002')
raw_text = Path('../data/sample/42002.txt').read_text()
df = parse_ndbc_text(raw_text, station_id=station.id)
print(f'Parsed {len(df)} rows; columns: {sorted(df.columns)}')
df.head()

## 2. Validate

In [ ]:
df = normalize_units(df)
df_qc, report = validate_dataframe(df, station_id=station.id)
print(f'Total obs: {report.n_total}')
print(f'Monotonic timestamps: {report.monotonic_timestamps}')
print(f'Gaps >= 1.5 h: {len(report.gaps_hours)}  max={max(report.gaps_hours, default=0):.1f} h')
report.to_dict()

## 3. QC flag breakdown

In [ ]:
qc_cols = [c for c in df_qc.columns if c.endswith('_qc')]
summary = pd.DataFrame({
    'good':       [(df_qc[c] == 1).sum() for c in qc_cols],
    'suspect':    [(df_qc[c] == 2).sum() for c in qc_cols],
    'missing':    [(df_qc[c] == 9).sum() for c in qc_cols],
}, index=[c[:-3] for c in qc_cols])
summary

## 4. Write daily NetCDFs + SHA-256

In [ ]:
files = write_daily_netcdf(df_qc, station, Path('../archive/daily'))
for path, digest in files:
    print(f'{path.name}  sha256={digest[:16]}...')

## 5. Inspect a produced NetCDF

In [ ]:
ds = xr.open_dataset(files[0][0])
print('Conventions:', ds.attrs['Conventions'])
print('Platform   :', ds.attrs['platform'])
print('Variables  :', list(ds.data_vars)[:6], '...')
ds['wind_speed'].plot(figsize=(11, 3))
plt.title(f'{station.name} — wind speed (one day)')
plt.show()
ds.close()

## 6. End-to-end via `Pipeline`

Everything above is wrapped by the `Pipeline` class. In production:

```python
from gbe import Pipeline
pipe = Pipeline()
results = pipe.run()        # pulls live NDBC + TABS
pipe.write_qc_report(results)
pipe.metrics.write('archive/metrics.prom')
```

Or invoked from the shell:

```bash
./bin/nightly.sh
```

which is the production cron/systemd entrypoint.